# NB12 — Orthogonal evidence and the exploratory spatial analysis (H3)

| Source | Question | Status |
|---|---|---|
| Human Protein Atlas single-cell-type RNA (normal tissue) | Are M genes neuron-restricted in *normal* brain (fits admixture) or also expressed in glia? Compared with R. | run |
| Ivy GAP RNA-seq (laser-microdissected regions) | H3: is M - R higher in infiltrating tumour (IT) / leading edge (LE) than in cellular tumour (CT)? | run; ~10 tumours, exploratory |
| CPTAC, DepMap, Open Targets | protein / dependency | **not run**: not informative for a tissue-level program; no verified endpoint |

No wet-lab evidence is claimed. H3 failures are recorded as `not_run` and never stop the pipeline.

In [ ]:
import os, sys, json
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
stop_if_smoke("NB12 needs the full results.")
verify_frozen()
OUT = P("orthogonal"); RAW = P("raw")
M = pd.read_csv(P("final", "M_genes.csv")).gene.astype(str).tolist(); R = pd.read_csv(P("final", "R_genes.csv")).gene.astype(str).tolist()
status = {}

In [ ]:
# ---- A. HPA single-cell-type RNA (normal tissues) ---------------------------------------------------------
try:
    zp = RAW / "HPA" / "rna_single_cell_type.tsv.zip"
    require_files([zp], "HPA file missing; re-run NB01.")
    h = pd.read_csv(zp, sep="\t", compression="zip")
    c_gene = find_col(h, ["Gene name", "gene_name"]); c_ct = find_col(h, ["Cell type", "cell_type"])
    c_val = find_col(h, ["nTPM", "nTPM_value", "TPM"])
    ct = h[c_ct].astype(str).str.lower()
    print("HPA columns used:", c_gene, c_ct, c_val)
    neuron = ct.str.contains("neuron")
    glia = ct.str.contains("astrocyt|oligodendro|microgli")
    print("Brain cell types found:", sorted(h.loc[neuron | glia, c_ct].unique()))
    require(neuron.any() and glia.any(), "HPA file has no neuron or glial cell types under the expected names.")
    w = h[neuron | glia].pivot_table(index=c_gene, columns=c_ct, values=c_val, aggfunc="max")
    neuron_cols = [c for c in w.columns if "neuron" in c.lower()]
    glia_cols = [c for c in w.columns if c not in neuron_cols]
    rows, per_gene = [], []
    for label, genes in [("M", M), ("R", R)]:
        sub = w.loc[w.index.intersection(genes)]
        nmax = sub[neuron_cols].max(axis=1); gmax = sub[glia_cols].max(axis=1)
        ratio = (nmax + 1) / (gmax + 1)
        per_gene.append(pd.DataFrame({"gene": sub.index, "set": label, "neuron_max_nTPM": nmax.values, "glia_max_nTPM": gmax.values, "neuron_to_glia": ratio.values}))
        rows.append({"set": label, "n_genes_in_HPA": len(sub), "median_neuron_to_glia_ratio": float(ratio.median()),
                     "fraction_neuron_restricted_ratio_gt_5": float((ratio > 5).mean()),
                     "fraction_expressed_in_glia_nTPM_gt_1": float((gmax > 1).mean())})
    pg = pd.concat(per_gene); pg.to_csv(OUT / "HPA_neuron_vs_glia_per_gene.csv", index=False)
    mw = stats.mannwhitneyu(pg[pg.set == "M"].neuron_to_glia, pg[pg.set == "R"].neuron_to_glia)
    hp = pd.DataFrame(rows); hp["mannwhitney_p_M_vs_R"] = mw.pvalue
    hp.to_csv(OUT / "HPA_neuron_vs_glia_M_vs_R.csv", index=False)
    print(hp.round(3).to_string(index=False))
    status["HPA"] = "run"
except Exception as exc:          # noqa: BLE001 (optional evidence; recorded, never fatal)
    print("HPA step not run:", exc); status["HPA"] = f"not_run: {str(exc)[:200]}"

In [ ]:
# ---- B. Ivy GAP (H3, exploratory) -------------------------------------------------------------------------
def ivy_region(name):
    s = str(name).lower()
    if any(k in s for k in ["microvascular", "pseudopalisad", "perinecrotic", "hyperplastic", "mvp", "pan", "pnz", "hbv", "necro"]):
        return None
    if s.startswith("le") or "leading edge" in s:
        return "LE"
    if s.startswith("it") or "infiltrating" in s:
        return "IT"
    if s.startswith("ct") or "cellular tumor" in s or "cellular tumour" in s:
        return "CT"
    return None


try:
    ivy_dir = RAW / "IvyGAP" / "extracted"
    files = sorted(p for p in ivy_dir.rglob("*") if p.is_file()) if ivy_dir.exists() else []
    require(bool(files), "Ivy GAP files not found.")
    print("Ivy GAP files:", [p.name for p in files])
    fp = next(p for p in files if "fpkm" in p.name.lower())
    cs = next(p for p in files if p.name.lower().startswith("columns"))
    rg = next(p for p in files if p.name.lower().startswith("rows"))
    expr = pd.read_csv(fp, index_col=0)
    rows_md = pd.read_csv(rg); cols_md = pd.read_csv(cs)
    print("rows-genes columns:", list(rows_md.columns)); print("columns-samples columns:", list(cols_md.columns))
    gid = find_col(rows_md, ["gene_id", "row_id", "id"]); gsym = find_col(rows_md, ["gene_symbol", "gene_name", "symbol"])
    gmap = dict(zip(rows_md[gid].astype(str), rows_md[gsym].astype(str)))
    expr.index = expr.index.astype(str)
    if expr.index.isin(list(gmap)).mean() > 0.9:
        expr.index = expr.index.map(gmap)
    else:
        require(len(expr) == len(rows_md), "Cannot match Ivy GAP rows to genes.")
        expr.index = rows_md[gsym].astype(str).values
    expr = expr[~expr.index.duplicated()]
    wid = find_col(cols_md, ["rna_well_id", "well_id", "sample_id"], required=False)
    expr.columns = expr.columns.astype(str)
    if wid and expr.columns.isin(cols_md[wid].astype(str)).mean() > 0.9:
        cmd = cols_md.set_index(cols_md[wid].astype(str)).loc[expr.columns]
    else:
        require(expr.shape[1] == len(cols_md), "Cannot match Ivy GAP columns to samples.")
        cmd = cols_md
    scol = find_col(cols_md, ["structure_abbreviation", "structure_acronym", "structure_name", "structure"])
    tcol = find_col(cols_md, ["tumor_name", "donor_name", "tumor_id", "donor_id"])
    print("Region labels (first 15):", cmd[scol].value_counts().head(15).to_dict())
    lg = np.log2(expr.clip(lower=0) + 1)
    lg = lg[lg.std(axis=1) > 0]
    z = zscore_df(lg, axis=1)
    d = pd.DataFrame({"sample": expr.columns, "structure_label": cmd[scol].astype(str).values, "tumor": cmd[tcol].astype(str).values,
                      "M": score_geneset(z, M).values, "R": score_geneset(z, R).values})
    d["region"] = d.structure_label.map(ivy_region)
    d["diff_M_minus_R"] = d.M - d.R
    keep = d.dropna(subset=["region"]).copy()
    print("Samples by region:", keep.region.value_counts().to_dict(), "| tumours:", keep.tumor.nunique())
    require(keep.tumor.nunique() >= 5 and set(keep.region) >= {"CT", "IT", "LE"}, "Fewer than 5 tumours or a region missing.")
    import statsmodels.formula.api as smf
    keep["region"] = pd.Categorical(keep.region, categories=["CT", "IT", "LE"])
    mdl = smf.mixedlm("diff_M_minus_R ~ C(region)", keep, groups=keep["tumor"]).fit(reml=True)
    print(mdl.summary())
    co = pd.DataFrame({"coef": mdl.params, "se": mdl.bse, "p": mdl.pvalues}).reset_index().rename(columns={"index": "term"})
    contr = co.term.str.contains(r"\[T\.(?:IT|LE)\]")
    co.loc[contr, "p_BH_over_IT_LE"] = multipletests(co.loc[contr, "p"], method="fdr_bh")[1]
    co.to_csv(OUT / "H3_IvyGAP_mixed_model_M_minus_R.csv", index=False)
    keep.to_csv(OUT / "H3_IvyGAP_scores.csv", index=False)
    print(co.round(4).to_string(index=False))
    fig, ax = plt.subplots(figsize=(4.6, 4))
    rng = np.random.default_rng(1)
    for i, s in enumerate(["CT", "IT", "LE"]):
        v = keep[keep.region == s].diff_M_minus_R
        ax.scatter(rng.normal(i, .06, len(v)), v, s=14, c="#1f3a93", alpha=.7)
        ax.hlines(v.median(), i - .25, i + .25, colors="#c0392b")
    ax.set_xticks(range(3)); ax.set_xticklabels(["cellular tumour", "infiltrating", "leading edge"])
    ax.set_ylabel("M score - R score (z)"); ax.set_title(f"H3 (exploratory, {keep.tumor.nunique()} tumours)")
    savefig(fig, "H3_IvyGAP_M_minus_R", folder="orthogonal"); plt.close(fig)
    le = co[co.term.str.contains(r"T\.LE")]; it = co[co.term.str.contains(r"T\.IT")]
    status["IvyGAP_H3"] = (f"run (exploratory): LE vs CT coef={le.coef.iloc[0]:.3f} BH p={le.p_BH_over_IT_LE.iloc[0]:.3g}; "
                           f"IT vs CT coef={it.coef.iloc[0]:.3f} BH p={it.p_BH_over_IT_LE.iloc[0]:.3g}")
except Exception as exc:     # noqa: BLE001 (H3 is exploratory and must not stop the pipeline)
    print("H3 not run:", exc); status["IvyGAP_H3"] = f"not_run: {str(exc)[:300]}"

status["CPTAC_DepMap_OpenTargets"] = "not_run: not informative for a tissue-level program; no verified automatic endpoint"
(OUT / "orthogonal_status.json").write_text(json.dumps(status, indent=2))
manifest_update("orthogonal", "status", status)
print("\nStatus:", json.dumps(status, indent=1))
print("\nNB12 complete. Next: NB13 (sensitivity).")